# QLoRA: fine-tuning a 1.1B-parameter LLM on a free Colab GPU

Instruction-tunes **TinyLlama-1.1B-Chat** with **QLoRA**: the base model is loaded in 4-bit NF4, and only small LoRA adapters (16-bit) are trained.

**Run on:** Google Colab, T4 GPU (`Runtime → Change runtime type → T4 GPU`). `bitsandbytes` 4-bit quantization needs an NVIDIA GPU, so this notebook is not part of the CPU study in this repo.

**Measured in this run (outputs below are from the actual Colab session):**

| | Value |
|---|---|
| GPU memory after loading the 4-bit model | 1.07 GB |
| Trainable parameters | 4.5M (0.41% of TinyLlama's 1.1B; PEFT reports 0.73% because 4-bit layers are counted in packed form) |
| Training | 1 epoch, 1,000 examples, 63 steps, ~5 min on a T4 |
| Adapter size | 17.2 MB |

Sections: theory (LoRA, QLoRA) → setup → 4-bit loading → QLoRA training → before/after generation → merging the adapter for deployment.

## 1. Why Not Just Fine-Tune Everything?

Traditional **full fine-tuning** updates *every* parameter of a pre-trained model. For a model with $N$ parameters, training with the Adam optimizer requires storing, per parameter:

- The weight itself (~2–4 bytes)
- A gradient (~2–4 bytes)
- Two optimizer moment terms, $m$ and $v$ (~4–8 bytes each)

For a 7-billion parameter model in 16-bit precision, this works out to roughly **70–110 GB** of GPU memory just for training state — far beyond what a single consumer or free-tier Colab GPU (often 16 GB) can hold.

**Parameter-Efficient Fine-Tuning (PEFT)** methods solve this by freezing the pre-trained weights and training only a small number of *additional* parameters. LoRA and QLoRA are the two most widely used PEFT techniques today.

## 2. LoRA: Low-Rank Adaptation

**Core idea:** when a model is fine-tuned for a new task, the *update* to each weight matrix tends to have a much lower "intrinsic rank" than the matrix itself. Instead of learning a full update $\Delta W$, LoRA learns a low-rank approximation of it.

For a pre-trained weight matrix $W_0 \in \mathbb{R}^{d \times k}$, LoRA freezes $W_0$ and represents the update as:

$$\Delta W = B A, \quad B \in \mathbb{R}^{d \times r},\ A \in \mathbb{R}^{r \times k},\ r \ll \min(d, k)$$

The forward pass becomes:

$$h = W_0 x + \Delta W x = W_0 x + \frac{\alpha}{r} B A\, x$$

Only $A$ and $B$ are trained; $W_0$ never changes. $\alpha$ is a scaling factor and $r$ is the **rank** — the key hyperparameter controlling capacity vs. efficiency.

**Why this saves so much:** for a $4096 \times 4096$ weight matrix (~16.8M parameters), a rank-8 LoRA adapter only needs $A$ ($8 \times 4096$) + $B$ ($4096 \times 8$) = **65,536 parameters** — about **0.4%** of the original matrix.

Key LoRA hyperparameters you'll see in code:
- `r`: the rank of the decomposition (typically 4–64)
- `lora_alpha`: scaling factor (often set to `2×r`)
- `target_modules`: which weight matrices to attach adapters to (usually attention projections)
- `lora_dropout`: dropout applied to the adapter path during training

## 3. QLoRA: LoRA + Quantization

QLoRA extends LoRA with one more idea: **load the frozen base model in 4-bit precision instead of 16-bit**, then train LoRA adapters (kept in 16-bit) on top of it. This shrinks the memory needed for the frozen weights by ~4x, while training quality stays close to full-precision fine-tuning.

QLoRA introduces three techniques:

1. **4-bit NormalFloat (NF4) quantization** — a data type designed for normally-distributed neural network weights, more accurate than standard 4-bit integer quantization.
2. **Double quantization** — the quantization constants themselves are quantized again, saving a small amount of additional memory.
3. **Paged optimizers** — uses NVIDIA unified memory to avoid out-of-memory crashes from occasional gradient-checkpointing memory spikes.

The result: models up to ~65B parameters can be fine-tuned on a single 48GB GPU, and smaller models (1–13B) comfortably fit on free-tier Colab GPUs.

## Quick Comparison

| | Full Fine-Tuning | LoRA | QLoRA |
|---|---|---|---|
| Base weights | Trainable | Frozen (16-bit) | Frozen (4-bit) |
| New parameters trained | All | Small adapters (~0.1–1%) | Small adapters (~0.1–1%) |
| GPU memory (7B model, approx.) | ~70–110 GB | ~16–20 GB | ~6–10 GB |
| Training speed | Baseline | Slightly faster (fewer gradients) | Slower than LoRA (dequantization overhead) |
| Output | New full model | Small adapter file (MBs) | Small adapter file (MBs) |
| Best for | Small models, max quality | Mid-size models, good GPUs | Large models, limited GPU memory |

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total GPU memory: {:.1f} GB".format(torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("⚠️ No GPU detected. Go to Runtime → Change runtime type → GPU, then re-run this cell.")

CUDA available: True
GPU: Tesla T4
Total GPU memory: 15.6 GB


In [2]:
!pip install -q -U transformers accelerate peft bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.4 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, PeftModel, TaskType, prepare_model_for_kbit_training

print("All libraries imported successfully ✅")

All libraries imported successfully ✅


In [ ]:
def print_trainable_parameters(model):
    """Print how many parameters are actually being trained."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    pct = 100 * trainable / total
    print(f"Trainable params: {trainable:,} | Total params: {total:,} | Trainable %: {pct:.4f}%")
    return trainable, total, pct


def folder_size_mb(path):
    return sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk(path) for f in fs) / 1024 ** 2

## QLoRA for instruction fine-tuning

Now for the real power move: fine-tuning a **1.1-billion-parameter language model** with QLoRA, comfortably on a free Colab GPU. We'll use **TinyLlama-1.1B-Chat** and a small instruction dataset.

In [14]:
instruct_ds = load_dataset("mlabonne/guanaco-llama2-1k", split="train")
print(instruct_ds)
print("\nExample:\n", instruct_ds[0]["text"])

README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 1000
})

Example:
 <s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocido mejor. En este caso, mi primera opción sería hablar con otros profesionales médicos, echar currículos en hospitales y cualquier centro de salud. En paralelo, trabajaría por mejorar mi marca personal como médico mediante un blog o formas digitales de comunicación como los vídeos. Y, para mejorar las posibilidades de encontrar trabajo, también participaría en congresos y encuentros para conseguir más contactos. Y, además de todo lo anterior, seguiría estudiando para presentarme a las oposiciones y ejercer la medicina en el sector público de mi país. </s>


In [15]:
model_name_b = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer_b = AutoTokenizer.from_pretrained(model_name_b)
tokenizer_b.pad_token = tokenizer_b.eos_token  # TinyLlama has no pad token by default

base_model_b = AutoModelForCausalLM.from_pretrained(
    model_name_b,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"GPU memory used after loading 4-bit model: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPU memory used after loading 4-bit model: 1.07 GB


In [16]:
base_model_b = prepare_model_for_kbit_training(base_model_b)

lora_config_b = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

lora_model_b = get_peft_model(base_model_b, lora_config_b)
_, _, qlora_trainable_pct = print_trainable_parameters(lora_model_b)

Trainable params: 4,505,600 | Total params: 620,111,872 | Trainable %: 0.7266%


In [17]:
def tokenize_b(batch):
    return tokenizer_b(batch["text"], truncation=True, max_length=512)

tokenized_instruct_ds = instruct_ds.map(tokenize_b, batched=True, remove_columns=instruct_ds.column_names)
collator_b = DataCollatorForLanguageModeling(tokenizer_b, mlm=False)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [18]:
training_args_b = TrainingArguments(
    output_dir="./tinyllama-qlora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    fp16=True,
    optim="paged_adamw_8bit",
    save_strategy="no",
    report_to=[],
)

# Note: if you get an "unexpected keyword argument" error here, it usually means
# your installed `transformers` version renamed a TrainingArguments field.
# Check the current docs for the equivalent argument name and re-run.

trainer_b = Trainer(
    model=lora_model_b,
    args=training_args_b,
    train_dataset=tokenized_instruct_ds,
    data_collator=collator_b,
)

trainer_b.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,1.658825
20,1.625677
30,1.604386
40,1.529566
50,1.561902
60,1.510451


TrainOutput(global_step=63, training_loss=1.574878700195797, metrics={'train_runtime': 309.0467, 'train_samples_per_second': 3.236, 'train_steps_per_second': 0.204, 'total_flos': 3057530689732608.0, 'train_loss': 1.574878700195797, 'epoch': 1.0})

In [19]:
adapter_dir_b = "./tinyllama-qlora-adapter"
lora_model_b.save_pretrained(adapter_dir_b)
print(f"QLoRA adapter size: {folder_size_mb(adapter_dir_b):.2f} MB")

QLoRA adapter size: 17.22 MB


### Before vs. After: Does the adapter actually change behavior?

PEFT models let us temporarily **disable** the adapter to compare the base model's output against the fine-tuned output — without reloading anything.

In [20]:
prompt = "[INST] What are three tips for staying focused while studying? [/INST]"
inputs_b = tokenizer_b(prompt, return_tensors="pt").to(lora_model_b.device)

lora_model_b.eval()

print("=== BEFORE fine-tuning (adapter disabled) ===")
with torch.no_grad(), lora_model_b.disable_adapter():
    out_before = lora_model_b.generate(**inputs_b, max_new_tokens=80, do_sample=False)
print(tokenizer_b.decode(out_before[0], skip_special_tokens=True))

print("\n=== AFTER fine-tuning (adapter enabled) ===")
with torch.no_grad():
    out_after = lora_model_b.generate(**inputs_b, max_new_tokens=80, do_sample=False)
print(tokenizer_b.decode(out_after[0], skip_special_tokens=True))

[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== BEFORE fine-tuning (adapter disabled) ===


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[INST] What are three tips for staying focused while studying? [/INST]

[Teacher] (voiceover) Remember to take breaks, stay hydrated, and avoid distractions. Staying focused is key to achieving your goals.

[Student] (voiceover) And don't forget to set realistic goals and celebrate small wins.

[Teacher] (voiceover) Good luck on your studies

=== AFTER fine-tuning (adapter enabled) ===
[INST] What are three tips for staying focused while studying? [/INST] Here are three tips for staying focused while studying:

1. Create a study schedule: Set aside specific times for studying and stick to it. This will help you avoid distractions and make the most of your time.

2. Use distractions: If you find yourself getting distracted, try to redirect your attention to something else. For example, you could listen


### Bonus: Merging Adapters for Deployment

For production, you may want a single standalone model file instead of a base model + adapter pair. This requires reloading the base model in **16-bit** (not 4-bit) — merging requires de-quantized weights — then fusing the adapter in.

⚠️ This loads a second copy of the model and is optional; skip if you're low on GPU memory.

In [21]:
merge_base = AutoModelForCausalLM.from_pretrained(
    model_name_b, torch_dtype=torch.float16, device_map="auto"
)
merge_model = PeftModel.from_pretrained(merge_base, adapter_dir_b)
merged_model = merge_model.merge_and_unload()

merged_model.save_pretrained("./tinyllama-merged")
tokenizer_b.save_pretrained("./tinyllama-merged")
print("Merged model saved — this can now be loaded with from_pretrained() like any normal model.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved — this can now be loaded with from_pretrained() like any normal model.


## ✅ Key Takeaways

- Full fine-tuning updates every parameter and needs huge amounts of GPU memory for optimizer state.
- **LoRA** freezes the base model and learns a low-rank update ($BA$) for selected weight matrices — often training under 1% of the original parameters.
- **QLoRA** adds 4-bit quantization of the frozen base model on top of LoRA, enabling fine-tuning of billion-parameter models on a single consumer GPU.
- Adapters are tiny (a few MB) and can be swapped in and out of a single frozen base model for different tasks.
- Merging fuses adapter weights into the base model for simpler, dependency-free deployment.

## 📚 Further Reading
- Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models* (2021)
- Dettmers et al., *QLoRA: Efficient Finetuning of Quantized LLMs* (2023)
- Hugging Face PEFT documentation: huggingface.co/docs/peft
- Hugging Face Transformers documentation: huggingface.co/docs/transformers